# 生成对抗网络 Generative Adversarial Networks

<img src="https://raw.githubusercontent.com/LisonEvf/practicalAI-cn/master/images/logo.png" width=150>

生成对抗网络（GANs）是由 Ian Goodfellow 等人在 2014 年提出的革命性生成模型。GANs 由两个对抗网络组成：生成器和判别器，它们相互博弈学习数据分布。

Generative Adversarial Networks (GANs) were introduced by Ian Goodfellow et al. in 2014 as a revolutionary generative model. GANs consist of two adversarial networks: a generator and a discriminator that learn data distributions through playing against each other.

<img src="https://raw.githubusercontent.com/LisonEvf/practicalAI-cn/master/images/gan.png" width=500>

# 概述 Overview

* **目标:**  学习真实数据分布，生成与真实数据相似的新样本。
* **优点:** 
  * 生成逼真的图像和其他数据
  * 无需显式建模数据分布
  * 可用于数据增强
* **缺点:**
  * 训练不稳定（模式崩溃）
  * 难以评估
* **其他:** 
  * 启发了许多变体：DCGAN, WGAN, StyleGAN 等
  * 广泛用于图像生成、风格迁移

# 设置 Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np

# 生成器 Generator

In [ ]:
class Generator(nn.Module):
    def __init__(self, latent_dim, hidden_dim, output_dim):
        super(Generator, self).__init__()
        
        self.net = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, hidden_dim * 2),
            nn.BatchNorm1d(hidden_dim * 2),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim * 2, hidden_dim * 4),
            nn.BatchNorm1d(hidden_dim * 4),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim * 4, output_dim),
            nn.Tanh()  # 输出范围 [-1, 1]
        )
    
    def forward(self, z):
        return self.net(z)

# 判别器 Discriminator

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(Discriminator, self).__init__()
        
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim * 4),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim * 4, hidden_dim * 2),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()  # 输出概率 [0, 1]
        )
    
    def forward(self, x):
        return self.net(x)

# 训练 Training

In [ ]:
# 超参数 Hyperparameters
latent_dim = 100
hidden_dim = 128
output_dim = 784  # 28x28 MNIST
learning_rate = 0.0002
num_epochs = 10
batch_size = 64


In [ ]:
# 加载数据 Load data
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])  # 标准化到 [-1, 1]
])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 初始化网络 Initialize networks
generator = Generator(latent_dim, hidden_dim, output_dim).to(device)
discriminator = Discriminator(output_dim, hidden_dim).to(device)


In [ ]:
# 优化器 Optimizers
g_optim = optim.Adam(generator.parameters(), lr=learning_rate, betas=(0.5, 0.999))
d_optim = optim.Adam(discriminator.parameters(), lr=learning_rate, betas=(0.5, 0.999))

# 损失函数 Loss function
criterion = nn.BCELoss()


In [ ]:
# 训练循环 Training loop
for epoch in range(num_epochs):
    for batch_idx, (real_images, _) in enumerate(train_loader):
        batch_size = real_images.size(0)
        real_images = real_images.view(-1, output_dim).to(device)
        
        # 真实标签和假标签 Real and fake labels
        real_labels = torch.ones(batch_size, 1).to(device)
        fake_labels = torch.zeros(batch_size, 1).to(device)
        
        # ========== 训练判别器 Train Discriminator ==========
        d_optim.zero_grad()
        
        # 判别真图像 Discriminate real images
        real_output = discriminator(real_images)
        d_loss_real = criterion(real_output, real_labels)
        
        # 生成假图像 Generate fake images
        noise = torch.randn(batch_size, latent_dim).to(device)
        fake_images = generator(noise)
        
        # 判别假图像 Discriminate fake images
        fake_output = discriminator(fake_images.detach())
        d_loss_fake = criterion(fake_output, fake_labels)
        
        # 总判别器损失 Total discriminator loss
        d_loss = d_loss_real + d_loss_fake
        d_loss.backward()
        d_optim.step()
        
        # ========== 训练生成器 Train Generator ==========
        g_optim.zero_grad()
        
        # 重新生成假图像 Regenerate fake images
        noise = torch.randn(batch_size, latent_dim).to(device)
        fake_images = generator(noise)
        fake_output = discriminator(fake_images)
        
        # 生成器损失（希望骗过判别器）Generator loss
        g_loss = criterion(fake_output, real_labels)
        g_loss.backward()
        g_optim.step()
    
    print(f'Epoch [{epoch+1}/{num_epochs}] - D_loss: {d_loss:.4f}, G_loss: {g_loss:.4f}')

In [ ]:
# 可视化生成图像 Visualize generated images
with torch.no_grad():
    noise = torch.randn(16, latent_dim).to(device)
    generated_images = generator(noise).cpu().view(-1, 1, 28, 28)
    
    fig, axes = plt.subplots(4, 4, figsize=(8, 8))
    for i, ax in enumerate(axes.flat):
        ax.imshow(generated_images[i].squeeze(), cmap='gray')
        ax.axis('off')
    plt.tight_layout()
    plt.show()

# TODO

- 条件 GAN Conditional GANs
- Wasserstein GAN (WGAN)
- 渐进式增长 GAN Progressive Growing GANs
- 图像到图像翻译 Image-to-Image Translation